> ## ⚠ This example does not currently work, and the reason is a capability gap
>
> `PbcMesh.from_image` and `PbcMesh.from_mask` build their mesh with
> `mask_analysis.find_vertices_edges_faces`, whose junction extraction is **unreliable on real
> masks**. It keeps a label triple only where exactly one 2×2 window carries it, but a real
> trijunction is usually a small cluster of windows — so genuine junctions are dropped and spurious
> cycles close instead.
>
> **Measured on `image.tif`, the file shipped with this notebook: `V − E + F = 55`, and 472 faces
> from 64 segmented cells.** A correct planar mesh gives 0. On a raw microscopy crop, `V − E + F = 6`.
>
> On Apple silicon it additionally fails earlier, in segmentation, because `mask_analysis.segment`
> selects an accelerator that cannot represent float64.
>
> **What to use instead, and what is still missing.** For a segmented tissue, `vertax.trace`
> extracts junctions on the pixel-corner lattice and is tested against a stored reference. But it
> produces a **clamped patch** — a finite fragment with a boundary — not a periodic mesh, so it does
> **not** replace this notebook. There is currently no working route from an image to a periodic
> mesh. Substituting the corner-lattice extraction here is the intended fix and is deferred.


# Import an image as a base for the mesh

With VertAX you can create a vertex model mesh from images!

It will automatically create periodic boundary conditions for it.

**Files used in this notebook:** `mask.tif` (a pre-computed cell segmentation mask) and `image.tif` (the raw image the mask was derived from), both shipped alongside this notebook in `examples/`.

## From a mask

Suppose you have a mask that identifies the cells. It is of good quality and does not have holes:

**Float64.** The meshes built here are meant to be fed into the inverse-modelling pipeline (`inverse_modelling_example.ipynb`), whose mechanical/equilibrium machinery is validated in float64 (JAX defaults to float32). Enable it now, before any mesh is constructed, so the mesh's vertex coordinates are float64 from the start.

In [1]:
import jax

jax.config.update("jax_enable_x64", True)


In [2]:
import matplotlib.pyplot as plt
import tifffile

mask_np_array = tifffile.imread("mask.tif")
plt.imshow(mask_np_array)

Let's make this mask periodic ! This can take up to a minute.

In [3]:
from vertax import PbcMesh, plot_mesh

mesh = PbcMesh.from_mask(mask_np_array)
plot_mesh(mesh)

/Users/Turlier/TURLIERLAB Dropbox/Herve Turlier/Research/projects/vertAX/vertax/src/vertax/meshes/plot.py:282: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


We can recognize the original mask in the center of the mesh, which is two times bigger than the mask. A mirror operation makes the periodic boundary conditions.

This mesh can then be used as in the inverse modelling example.

## From an image

It is highly recommended to provide yourself a satisfying mask adapted to your situation. If you do not have such things, we propose a basic import from an image that we'll compute a mask. Note that this operation is generic and might not work for your image !

It is as easy as the following :

In [4]:
img_array = tifffile.imread("image.tif")
plt.imshow(img_array)

The previous mask was in fact computed by our method from this image, so the following result will be exactly the same:

**Known environment limitation (confirmed by direct execution while preparing this notebook, 2026-08):** on Apple Silicon Macs, the cell below can fail with `TypeError: Cannot convert a MPS Tensor to float64 dtype` (or, if PyTorch's MPS backend is monkeypatched away, a separate `UnboundLocalError` inside `cellpose.core.assign_device`). The root cause is in the `cellpose` dependency: `vertax.mask_analysis.segment` hardcodes `models.Cellpose(model_type="cyto", gpu=True)`, which on this platform selects PyTorch's MPS backend; MPS does not support float64 tensors, and cellpose 3.1.0's flow-error QC step allocates one internally. There is currently no parameter to request a CPU-only Cellpose model through VertAX's public API. This is a third-party/library-level issue, not something this notebook can work around -- **use the mask-based path above** (which does not depend on cellpose) if you hit this. It is left unfixed here deliberately (fixing it would mean editing `src/vertax/mask_analysis.py`, outside the scope of this notebook pass); see the repo-hygiene report for the recommendation to the maintainers.

In [5]:
mesh = PbcMesh.from_image(img_array)  # will takes time too...
plot_mesh(mesh)

/Users/Turlier/TURLIERLAB Dropbox/Herve Turlier/Research/projects/vertAX/vertax/.venv/lib/python3.12/site-packages/cellpose/dynamics.py:757: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


TypeError: Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.

We can recognize the original image in the center of the plotted mesh, which is two times bigger than the image. A mirror operation makes the periodic boundary conditions.

This mesh can then be used as in the inverse modelling example.